# CVInsight — Week 2 Summary
**CV Parser & Text Extraction**  
Git tag: `v0.2`

---

## Week 2 Deliverables Checklist

| # | Deliverable | Status |
|---|---|---|
| 1 | `parse_cv()` handles PDF, DOCX, TXT | ✅ |
| 2 | OCR fallback for scanned PDFs | ✅ |
| 3 | `split_sections()` returns dict with section names | ✅ |
| 4 | `clean_cv_text()` normalizes encoding/whitespace/bullets | ✅ |
| 5 | Unit tests pass: `pytest tests/` | ✅ 133 passed |
| 6 | Tested on 20 real CVs — 85%+ success rate | ✅ 95% (19/20) |
| 7 | Git tag `v0.2` | ✅ |

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
os.chdir('..')  # ensure repo root
print('Working dir:', os.getcwd())

## 1. Parser — parse_cv()

In [ ]:
from src.parser.parser import parse_cv, get_supported_extensions
print('Supported formats:', get_supported_extensions())

## 2. Section Splitter — split_sections()

In [ ]:
from src.parser.section_splitter import split_sections

sample_cv = """John Doe
john@email.com | +880-1700-000000

EDUCATION
BSc Computer Science, BUET, 2020

EXPERIENCE
Software Engineer, Google, 2020 - 2023
  - Built ML pipelines

SKILLS
Python, SQL, Docker, TensorFlow

CERTIFICATIONS
AWS Certified Solutions Architect, 2022
"""

sections = split_sections(sample_cv)
print('Sections detected:', list(sections.keys()))
print()
for name, content in sections.items():
    print(f'[{name}]')
    print(content[:120])
    print()

## 3. Cleaner — clean_cv_text()

In [ ]:
from src.parser.cleaner import clean_cv_text

dirty = "John Doe\u00a0\n• Python\n\u2022 Docker\nPage 1 of 2\npro\ufb01le"
print('BEFORE:', repr(dirty))
print()
print('AFTER: ', repr(clean_cv_text(dirty)))

## 4. Real CV Test Results

In [ ]:
import pandas as pd

results = pd.read_csv('data/processed/parser_test_results.csv')
print(results[['cv_id','status','num_sections','sections_found']].to_string(index=False))
print()
n_pass = (results['status'] == 'PASS').sum()
print(f'Pass rate: {n_pass}/{len(results)} = {n_pass/len(results)*100:.1f}%')

## 5. Test Suite

In [ ]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/', '-v', '--tb=short'],
    capture_output=True, text=True
)
print(result.stdout[-3000:])  # last 3000 chars
if result.returncode == 0:
    print('\n✅ All tests passed')
else:
    print('\n❌ Some tests failed')
    print(result.stderr[-1000:])

## 6. Files Created This Week

```
src/parser/
├── pdf_parser.py        ✅ pdfplumber → pdfminer → pypdf waterfall
├── docx_parser.py       ✅ python-docx, preserves tables
├── txt_parser.py        ✅ utf-8/latin-1/cp1252 fallback chain
├── ocr_parser.py        ✅ pytesseract + pdf2image fallback
├── parser.py            ✅ unified parse_cv() entry point
├── section_splitter.py  ✅ 80+ heading aliases → canonical sections
└── cleaner.py           ✅ encoding/bullet/whitespace normalization

tests/
├── test_pdf_parser.py       ✅ 13 tests
├── test_parser.py           ✅ 24 tests
├── test_ocr_parser.py       ✅  9 tests (2 need Tesseract)
├── test_section_splitter.py ✅ 29 tests
└── test_cleaner.py          ✅ 39 tests

scripts/
└── test_parser_on_real_cvs.py  ✅ 19/20 passed (95%)
```

## 7. Lessons Learned

| # | Lesson | Day |
|---|---|---|
| 11 | Use `tempfile.gettempdir()` not `/tmp/` for cross-platform test paths | Day 8 |
| 12 | In text cleaning, strip trailing whitespace BEFORE collapsing blank lines | Day 8 |
| 13 | Always inspect one raw sample before writing a pipeline — dataset format assumptions are almost always wrong | Day 13 |

## 8. Next Week

**Week 3 — Information Extraction (NER)**
- `contact_extractor.py` — email, phone, LinkedIn
- `skill_extractor.py` — spaCy EntityRuler + skill_taxonomy.json  
- `education_extractor.py` — degree, institution, year, GPA
- `experience_extractor.py` — titles, companies, date ranges, duration_months
- `extractor.py` — master `extract_all(text) → CVSchema`
- Target: Skills F1 ≥ 0.75, 200-300 CVs processed